> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataVisualization/blob/main/notebooks/01b_Choosing_the_Right_Chart.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Choosing the Right Chart

**Series:** Data Visualization in Python (notebook 2 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Show distributions with histograms and box plots
- Show relationships with scatter plots, and why a log scale helps
- Replace crowded charts with small multiples
- Build a correlation heatmap correctly
- Avoid dual axes and overused pie charts
- Make an interactive, animated chart

**You should already know:** notebook 1a.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## Step 1: Import libraries and load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

gap = px.data.gapminder()
y2007 = gap[gap["year"] == 2007]

**Seaborn** builds on Matplotlib and makes statistical charts (box plots, small multiples, heatmaps)
with less code.

## Step 2: Distributions with a histogram

A histogram shows how values are spread out: where most countries are, and whether there are extremes.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(y2007["gdpPercap"], bins=30, color="#3b6ea5", edgecolor="white")
plt.title("Most countries had low GDP per person in 2007; a few were very rich")
plt.xlabel("GDP per person (US dollars)")
plt.ylabel("Number of countries")
plt.show()

The data are **skewed**: a long tail to the right. Skewed data are common in social science (income,
population, city size). The same issue affects how maps are classified, as you will see in notebook 1c.

## Step 3: Compare groups with a box plot

A box plot summarizes a distribution: the box spans the middle 50% of values, the line inside is the
median, and dots are unusual values.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=y2007, x="continent", y="lifeExp", color="#9ecae1")
plt.title("Life expectancy by continent, 2007")
plt.xlabel("")
plt.ylabel("Life expectancy (years)")
plt.show()

## Step 4: Relationships with a scatter plot

Is income related to life expectancy? A scatter plot puts one variable on each axis.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))

left.scatter(y2007["gdpPercap"], y2007["lifeExp"], alpha=0.7)
left.set_title("Linear scale: most points crowd on the left")

right.scatter(y2007["gdpPercap"], y2007["lifeExp"], alpha=0.7)
right.set_xscale("log")
right.set_title("Log scale: the relationship is clearer")

for ax in (left, right):
    ax.set_xlabel("GDP per person (US dollars)")
    ax.set_ylabel("Life expectancy (years)")
plt.show()

On a **log scale**, each step along the axis multiplies the value (1,000, 10,000, 100,000). It spreads
out skewed data. The pattern: life expectancy rises quickly as income grows from very low levels, then
levels off.

Remember: a relationship in a scatter plot is a **correlation**, not proof that one causes the other.

## Step 5: Add a third and fourth variable

Color can show a category and size can show a quantity. This is a **bubble chart**.

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.scatterplot(data=y2007, x="gdpPercap", y="lifeExp", hue="continent",
                size="pop", sizes=(20, 1200), alpha=0.6, palette="Set2")
plt.xscale("log")
plt.title("Income, life expectancy and population by country, 2007")
plt.xlabel("GDP per person (US dollars, log scale)")
plt.ylabel("Life expectancy (years)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.show()

`Set2` is a categorical palette that works for most people with color vision deficiency. Each color
is a continent; there is no order between them.

## Step 6: Small multiples instead of a crowded chart

Plotting all 142 countries on one chart makes a tangle. **Small multiples** split the data into many
small panels that share the same axes, so they are easy to compare.

In [ ]:
grid = sns.relplot(data=gap, x="year", y="lifeExp", col="continent", col_wrap=3,
                   kind="line", units="country", estimator=None,
                   color="#3b6ea5", alpha=0.35, linewidth=1, height=3)
grid.set_titles("{col_name}")
grid.set_axis_labels("Year", "Life expectancy (years)")
grid.figure.suptitle("Life expectancy by country, 1952 to 2007", y=1.03)
plt.show()

Each thin line is one country. Because the panels share axes, you can see at once that Africa's countries
are more spread out, and that a few lines in Africa dip sharply in the 1990s.

## Step 7: A correlation heatmap, done correctly

A heatmap colors a grid of numbers. It is most often used for a **correlation matrix**: how strongly each
pair of variables moves together, from -1 (opposite directions) through 0 (unrelated) to +1 (same
direction).

First we build a few variables per country.

In [ ]:
first = gap[gap["year"] == 1952].set_index("country")
last = gap[gap["year"] == 2007].set_index("country")

countries = pd.DataFrame({
    "life expectancy 2007": last["lifeExp"],
    "log GDP per person 2007": np.log10(last["gdpPercap"]),
    "log population 2007": np.log10(last["pop"]),
    "life expectancy gain since 1952": last["lifeExp"] - first["lifeExp"],
    "GDP growth since 1952 (x)": last["gdpPercap"] / first["gdpPercap"],
})
corr = countries.corr()
corr.round(2)

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            square=True, cbar_kws={"label": "Correlation"})
plt.title("How country indicators relate to each other")
plt.show()

Two things make this heatmap correct:

- It uses a **diverging** palette centered on 0 (`center=0`, `vmin=-1`, `vmax=1`), because 0 is a
  meaningful midpoint: red is positive, blue is negative, white is no relationship.
- The numbers are printed in each cell, so color is not the only way to read it.

A heatmap of raw values (for example test scores) shows **values**, not relationships; do not read its
darker cells as "stronger relationships".

## Step 8: Two measures, two panels (no dual axes)

It is tempting to put two different measures on one chart with a second y-axis. Avoid it: the reader
cannot tell which line goes with which axis, and the scales can be stretched to suggest any relationship.
Use two panels that share the x-axis instead.

In [ ]:
us = gap[gap["country"] == "United States"]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
top.plot(us["year"], us["lifeExp"], marker="o", color="#3b6ea5")
top.set_ylabel("Life expectancy\n(years)")
top.set_title("United States, 1952 to 2007")
bottom.plot(us["year"], us["gdpPercap"], marker="o", color="#e6550d")
bottom.set_ylabel("GDP per person\n(US dollars)")
bottom.set_xlabel("Year")
plt.show()

## Step 9: Pie charts, used sparingly

Pie charts show parts of a whole, but people compare angles poorly. They work only with two or three
parts. A bar chart is almost always easier to read.

In [ ]:
pop_share = y2007.groupby("continent")["pop"].sum().sort_values()
pop_share = 100 * pop_share / pop_share.sum()

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.pie(pop_share, labels=pop_share.index)
left.set_title("Pie: which slice is bigger, Africa or the Americas?")
right.barh(pop_share.index, pop_share.values, color="#3b6ea5")
right.set_xlabel("Share of world population in this dataset, 2007 (%)")
right.set_title("Bar: the comparison is immediate")
plt.show()

## Step 10: An interactive, animated chart

Plotly makes charts you can hover over, zoom and animate. This is the famous "Gapminder" chart: press
**Play**, or drag the slider.

In [ ]:
fig = px.scatter(gap, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                 hover_name="country", animation_frame="year", animation_group="country",
                 log_x=True, size_max=55, range_x=[100, 100000], range_y=[25, 90],
                 labels={"gdpPercap": "GDP per person (US$, log scale)",
                         "lifeExp": "Life expectancy (years)"},
                 title="Income and life expectancy, 1952 to 2007")
fig.show()

Interactive charts are great for exploring and for web pages. For a printed report, choose one or two
years and make a static chart.

## Which chart should I use?

| Question | Chart |
|---|---|
| How are values distributed? | Histogram, box plot |
| How do groups compare? | Bar chart, box plot |
| Are two variables related? | Scatter plot (log scale for skewed data) |
| How do many groups change over time? | Small multiples |
| How do many variables relate to each other? | Correlation heatmap |
| Two different measures over time? | Two panels with a shared x-axis |
| Parts of a whole? | Bar chart; pie only for 2 or 3 parts |

## Try it yourself

Make a **box plot of GDP per person by continent in 2007**, with a **log scale** on the y-axis. Which continent has the widest spread?

*Hint:* copy Step 3, change `y`, and add `plt.yscale("log")`.

<details>
<summary><b>Show a solution</b></summary>

```python
plt.figure(figsize=(8, 4))
sns.boxplot(data=y2007, x="continent", y="gdpPercap", color="#9ecae1")
plt.yscale("log")
plt.title("GDP per person by continent, 2007")
plt.xlabel("")
plt.ylabel("GDP per person (US dollars, log scale)")
plt.show()
# Asia and Africa have the widest spreads, from some of the poorest to some of the richest countries.
```

</details>

In [ ]:
# Write your answer here

## Recap

- Histograms and box plots show distributions; log scales tame skewed data.
- Scatter plots show relationships, which are correlations, not causes.
- Small multiples beat crowded single charts.
- Correlation heatmaps need a diverging palette centered on zero, with values printed.
- Use two panels instead of dual axes, and bars instead of most pie charts.

**Next:** [From Charts to Maps](01c_From_Charts_to_Maps.ipynb)